# BERT Models Embeddings Generation
This notebook encodes sentences from a .csv file using multiple BERT models:
- BERT (base)
- RoBERTa (base)
- NeoBERT
- ModernBERT

Embeddings are saved to separate CSV files for each model.


## Setup and Imports


In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check for CUDA availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Using device: cuda
GPU: Tesla T4


## Load Data


In [2]:
# Load the sentences dataset
df = pd.read_csv('/content/extreme_pole_sentences.csv')
print(f"Loaded {len(df)} sentences")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()


Loaded 90 sentences

Columns: ['sentence', 'pos/neg', 'V/A/D']

First few rows:


,sentence,pos/neg,V/A/D
0,The soft embrace of my lover fills me with pur...,pos,V
1,Sitting in church the peaceful sermon washes o...,pos,V
2,My friends kind words lift my spirits gently m...,pos,V
3,Watching the sunset with my family a wave of q...,pos,V
4,The taste of my favorite meal brings simple ca...,pos,V


## Helper Function for Encoding


In [3]:
def encode_sentences(model_name, sentences, device, batch_size=8):
    """
    Encode sentences using a specified model.

    Args:
        model_name: HuggingFace model identifier
        sentences: List of sentences to encode
        device: torch device (cpu/cuda)
        batch_size: Number of sentences to process at once

    Returns:
        numpy array of embeddings (shape: n_sentences x embedding_dim)
    """
    print(f"\nLoading {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    embeddings = []

    with torch.no_grad():
        for i in tqdm(range(0, len(sentences), batch_size), desc=f"Encoding with {model_name}"):
            batch = sentences[i:i+batch_size]

            # Tokenize
            inputs = tokenizer(batch, padding=True, truncation=True,
                             max_length=512, return_tensors='pt').to(device)

            # Get embeddings (use [CLS] token)
            outputs = model(**inputs)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embeddings)

    # Clear memory
    del model, tokenizer
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    return np.vstack(embeddings)


## Helper Function to Save Embeddings


In [4]:
def save_embeddings_to_csv(df, embeddings, model_name, output_path):
    """
    Save embeddings along with original columns to CSV.

    Args:
        df: Original dataframe with sentence, pos/neg, V/A/D columns
        embeddings: numpy array of embeddings
        model_name: Name of the model (for column naming)
        output_path: Path to save the CSV
    """
    # Create a new dataframe with original columns
    result_df = df.copy()

    # Add embedding dimensions as columns
    embedding_dim = embeddings.shape[1]
    for i in range(embedding_dim):
        result_df[f'emb_{i}'] = embeddings[:, i]

    # Save to CSV
    result_df.to_csv(output_path, index=False)
    print(f"Saved {len(result_df)} rows with {embedding_dim}-dimensional embeddings to {output_path}")


## 1. BERT (base-uncased)


In [5]:
# BERT
bert_embeddings = encode_sentences(
    model_name='bert-base-uncased',
    sentences=df['sentence'].tolist(),
    device=device
)

save_embeddings_to_csv(
    df=df,
    embeddings=bert_embeddings,
    model_name='bert',
    output_path='/content/extreme_pole_embeddings_bert.csv'
)



Loading bert-base-uncased...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Encoding with bert-base-uncased:   0%|          | 0/12 [00:00<?, ?it/s]

Saved 90 rows with 768-dimensional embeddings to /content/extreme_pole_embeddings_bert.csv


## 2. RoBERTa (base)


In [6]:
# RoBERTa
roberta_embeddings = encode_sentences(
    model_name='roberta-base',
    sentences=df['sentence'].tolist(),
    device=device
)

save_embeddings_to_csv(
    df=df,
    embeddings=roberta_embeddings,
    model_name='roberta',
    output_path='/content/extreme_pole_embeddings_roberta.csv'
)



Loading roberta-base...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Encoding with roberta-base:   0%|          | 0/12 [00:00<?, ?it/s]

Saved 90 rows with 768-dimensional embeddings to /content/extreme_pole_embeddings_roberta.csv


## 3. NeoBERT


In [9]:
!pip install xformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 9.9 MB/s eta 0:00:00


In [10]:
# NeoBERT (using the official model from HuggingFace)
neobert_embeddings = encode_sentences(
    model_name='chandar-lab/NeoBERT',
    sentences=df['sentence'].tolist(),
    device=device
)

save_embeddings_to_csv(
    df=df,
    embeddings=neobert_embeddings,
    model_name='neobert',
    output_path='/content/extreme_pole_embeddings_neobert.csv'
)



Loading chandar-lab/NeoBERT...
The repository chandar-lab/NeoBERT contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/chandar-lab/NeoBERT .
 You can inspect the repository content at https://hf.co/chandar-lab/NeoBERT.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


rotary.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/chandar-lab/NeoBERT:
- rotary.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/chandar-lab/NeoBERT:
- rotary.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


The repository chandar-lab/NeoBERT contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/chandar-lab/NeoBERT .
 You can inspect the repository content at https://hf.co/chandar-lab/NeoBERT.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


model.safetensors:   0%|          | 0.00/981M [00:00<?, ?B/s]

Encoding with chandar-lab/NeoBERT:   0%|          | 0/12 [00:00<?, ?it/s]

Saved 90 rows with 768-dimensional embeddings to /content/extreme_pole_embeddings_neobert.csv


## 4. ModernBERT


In [11]:
# ModernBERT (using the base model)
modernbert_embeddings = encode_sentences(
    model_name='answerdotai/ModernBERT-base',
    sentences=df['sentence'].tolist(),
    device=device
)

save_embeddings_to_csv(
    df=df,
    embeddings=modernbert_embeddings,
    model_name='modernbert',
    output_path='/content/extreme_pole_embeddings_modernbert.csv'
)



Loading answerdotai/ModernBERT-base...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Encoding with answerdotai/ModernBERT-base:   0%|          | 0/12 [00:00<?, ?it/s]

W1017 09:52:34.010000 278 torch/_inductor/utils.py:1436] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Saved 90 rows with 768-dimensional embeddings to /content/extreme_pole_embeddings_modernbert.csv


## Embedding Dimensions


In [12]:
print(f"Embedding dimensions:")
print(f"  - BERT: {bert_embeddings.shape[1]}")
print(f"  - RoBERTa: {roberta_embeddings.shape[1]}")
print(f"  - NeoBERT: {neobert_embeddings.shape[1]}")
print(f"  - ModernBERT: {modernbert_embeddings.shape[1]}")


Embedding dimensions:
  - BERT: 768
  - RoBERTa: 768
  - NeoBERT: 768
  - ModernBERT: 768
